# Lab 13: Discrete Fourier Transform, FFT, and Periodogram

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-13-dft-fft-periodogram-lab.ipynb)

This lab is designed for **independent study**. Before every programming step, you will find the mathematical and statistical background needed to understand what the code is doing.

In Chapter 13, we move from the population frequency-domain idea of Chapter 12 to finite data. Given one observed time series

$$
x_0,x_1,\ldots,x_{n-1},
$$

the **discrete Fourier transform** decomposes the data into finite-frequency components. The **FFT** computes this decomposition efficiently, and the **periodogram** uses the squared Fourier coefficients to estimate how much variation appears at each frequency.

## Learning goals

By the end of this lab, you should be able to:

1. Explain the difference between time-domain and frequency-domain representations.
2. Compute a DFT directly from its definition.
3. Compare a direct DFT with `numpy.fft.fft`.
4. Interpret Fourier frequencies measured in cycles per time step.
5. Compute and interpret a periodogram.
6. Recognize aliasing and spectral leakage.
7. Use smoothing and tapering to make periodograms easier to interpret.
8. Compare empirical periodograms with theoretical spectra of simple models.

## 0. Setup

We use only standard scientific Python packages: `numpy`, `pandas`, and `matplotlib`. This makes the lab easy to run in Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

## 1. Background: finite Fourier frequencies

For a length-$n$ sequence $x_0,\ldots,x_{n-1}$, the Fourier frequencies are

$$
\omega_k = \frac{k}{n}, \qquad k=0,1,\ldots,n-1.
$$

A frequency $\omega$ means **cycles per time step**. For example:

- $\omega=0.25$ means one cycle every $1/0.25=4$ time steps.
- $\omega=1/12$ means one cycle every $12$ time steps.
- $\omega=0$ is the constant or mean component.

In this lab, we use the normalized DFT convention

$$
c_k = \frac{1}{n}\sum_{t=0}^{n-1} x_t e^{-2\pi i k t/n}.
$$

The coefficient $c_k$ tells us how much of the sequence aligns with a complex sinusoid at frequency $k/n$.

## 2. Create a synthetic signal with hidden periodicities

We first generate a time series with two sinusoidal components plus noise:

$$
x_t = 2\sin(2\pi t/24) + 0.8\cos(2\pi t/8) + \varepsilon_t.
$$

The first component has period $24$, and the second has period $8$.
The goal is to see whether the DFT and periodogram can recover these periods from the observed data.

In [ ]:
n = 240
t = np.arange(n)

period_1 = 24
period_2 = 8

signal = 2.0 * np.sin(2 * np.pi * t / period_1) + 0.8 * np.cos(2 * np.pi * t / period_2)
noise = rng.normal(0, 0.8, size=n)
x = signal + noise

series = pd.DataFrame({
    "time": t,
    "signal_without_noise": signal,
    "observed_x": x
})

series.head()

In [ ]:
plt.figure()
plt.plot(t, x, label="observed series")
plt.plot(t, signal, label="signal without noise", linewidth=2)
plt.xlabel("time")
plt.ylabel("value")
plt.title("Synthetic time series with two hidden periods")
plt.legend()
plt.show()

### Checkpoint 1

Before computing the periodogram, answer these questions:

1. Which periods did we build into the data?
2. What are the corresponding frequencies?
3. Why might noise make the hidden periods hard to see in the time plot?

In [ ]:
freq_1 = 1 / period_1
freq_2 = 1 / period_2

print("Period 1:", period_1, "frequency:", freq_1)
print("Period 2:", period_2, "frequency:", freq_2)

## 3. Implement the DFT from scratch

The DFT is a matrix-vector multiplication. If

$$
E_{k,t}=e^{-2\pi i k t/n},
$$

then

$$
c_k = \frac{1}{n}\sum_{t=0}^{n-1} E_{k,t}x_t.
$$

The direct implementation costs roughly $O(n^2)$ operations. The FFT computes the same numbers much faster, in roughly $O(n\log n)$ operations.

We first implement the DFT directly for a small example so that the formula is transparent.

In [ ]:
def dft_direct(values):
    """Compute normalized DFT coefficients c_k = (1/n) sum_t x_t exp(-2 pi i k t / n)."""
    values = np.asarray(values, dtype=float)
    n_local = len(values)
    t_local = np.arange(n_local)
    k_local = t_local.reshape((n_local, 1))
    E = np.exp(-2j * np.pi * k_local * t_local / n_local)
    return E @ values / n_local

small_x = x[:16]
c_direct_small = dft_direct(small_x)
c_fft_small = np.fft.fft(small_x) / len(small_x)

max_difference = np.max(np.abs(c_direct_small - c_fft_small))
print("Maximum difference between direct DFT and FFT:", max_difference)

The maximum difference should be close to machine precision. That means `np.fft.fft(x) / n` computes the same normalized DFT coefficients as our direct formula.

In [ ]:
comparison = pd.DataFrame({
    "k": np.arange(8),
    "direct_real": np.real(c_direct_small[:8]),
    "direct_imag": np.imag(c_direct_small[:8]),
    "fft_real": np.real(c_fft_small[:8]),
    "fft_imag": np.imag(c_fft_small[:8])
})
comparison

## 4. Compute the periodogram

The periodogram measures the squared size of Fourier coefficients. With our convention, define

$$
I(\omega_k)=n|c_k|^2.
$$

Different textbooks use slightly different constants. For interpretation in this lab, constants are not the main issue. We care about the **locations of large peaks**.

For a real-valued time series, positive and negative frequencies carry symmetric information. We usually look only at frequencies from $0$ to $1/2$, where $1/2$ is the Nyquist frequency.

In [ ]:
def periodogram(values):
    values = np.asarray(values, dtype=float)
    n_local = len(values)
    centered = values - np.mean(values)
    coeffs = np.fft.fft(centered) / n_local
    freqs = np.fft.fftfreq(n_local, d=1.0)
    power = n_local * np.abs(coeffs) ** 2
    return freqs, power, coeffs

freqs, power, coeffs = periodogram(x)
positive = (freqs >= 0) & (freqs <= 0.5)

plt.figure()
plt.plot(freqs[positive], power[positive])
plt.xlabel("frequency: cycles per time step")
plt.ylabel("periodogram")
plt.title("Periodogram of the observed series")
plt.show()

### Checkpoint 2

The strongest peaks should appear near frequencies $1/24$ and $1/8$.

Because frequency can be hard to interpret, we convert frequency to period using

$$
\text{period}=\frac{1}{\text{frequency}}.
$$

The frequency $0$ corresponds to the mean and has no finite period.

In [ ]:
# Find the largest positive-frequency peaks, excluding frequency 0.
pos_freqs = freqs[positive]
pos_power = power[positive]
nonzero = pos_freqs > 0

peak_indices = np.argsort(pos_power[nonzero])[-8:][::-1]
peak_freqs = pos_freqs[nonzero][peak_indices]
peak_power = pos_power[nonzero][peak_indices]
peak_periods = 1 / peak_freqs

peaks = pd.DataFrame({
    "frequency": peak_freqs,
    "period": peak_periods,
    "periodogram": peak_power
})

peaks

## 5. Parseval identity: time-domain energy and frequency-domain energy

Fourier analysis is not only a visualization tool. It preserves the total squared size of a sequence.

With our normalized DFT convention,

$$
\frac{1}{n}\sum_{t=0}^{n-1} (x_t-\bar{x})^2
= \sum_{k=0}^{n-1}|c_k|^2.
$$

This is the finite-sample version of the idea that total variance can be distributed across frequencies.

In [ ]:
centered = x - np.mean(x)
coeffs_centered = np.fft.fft(centered) / n

time_domain_mean_square = np.mean(centered ** 2)
freq_domain_mean_square = np.sum(np.abs(coeffs_centered) ** 2)

print("Time-domain mean square:", time_domain_mean_square)
print("Frequency-domain sum of squared coefficients:", freq_domain_mean_square)
print("Absolute difference:", abs(time_domain_mean_square - freq_domain_mean_square))

## 6. Reconstruct the series from a few Fourier frequencies

A useful way to understand the DFT is to reconstruct the series using only the largest Fourier components. This is like keeping the strongest periodic patterns and discarding many weaker components.

The inverse FFT reconstructs the series from all Fourier coefficients. If we set most coefficients to zero before applying the inverse transform, we obtain a low-dimensional Fourier approximation.

In [ ]:
def reconstruct_from_top_frequencies(values, num_pairs=2):
    values = np.asarray(values, dtype=float)
    n_local = len(values)
    mean_value = np.mean(values)
    centered = values - mean_value
    F = np.fft.fft(centered)
    freqs_local = np.fft.fftfreq(n_local)

    # Select largest positive frequency powers, excluding 0 and Nyquist duplication.
    pos_mask = (freqs_local > 0) & (freqs_local < 0.5)
    powers_local = np.abs(F) ** 2
    candidate_indices = np.where(pos_mask)[0]
    selected_pos = candidate_indices[np.argsort(powers_local[candidate_indices])[-num_pairs:]]

    keep = np.zeros(n_local, dtype=bool)
    for idx in selected_pos:
        keep[idx] = True
        keep[-idx] = True  # conjugate negative frequency

    F_filtered = np.zeros_like(F)
    F_filtered[keep] = F[keep]
    reconstructed = np.fft.ifft(F_filtered).real + mean_value
    return reconstructed, selected_pos, freqs_local[selected_pos]

recon_1, idx_1, f_1 = reconstruct_from_top_frequencies(x, num_pairs=1)
recon_2, idx_2, f_2 = reconstruct_from_top_frequencies(x, num_pairs=2)
recon_4, idx_4, f_4 = reconstruct_from_top_frequencies(x, num_pairs=4)

print("Top 1 frequency:", f_1, "period:", 1 / f_1)
print("Top 2 frequencies:", f_2, "periods:", 1 / f_2)

In [ ]:
plt.figure()
plt.plot(t, x, label="observed", alpha=0.45)
plt.plot(t, signal, label="true signal", linewidth=2)
plt.plot(t, recon_1, label="top 1 frequency pair")
plt.plot(t, recon_2, label="top 2 frequency pairs")
plt.xlim(0, 120)
plt.xlabel("time")
plt.ylabel("value")
plt.title("Fourier reconstruction using a few dominant frequencies")
plt.legend()
plt.show()

### Checkpoint 3

1. Does the top 2 frequency reconstruction recover the main signal shape?
2. Which part of the observed series is not captured by the top Fourier frequencies?
3. Why might a small number of Fourier components be useful as features in a forecasting model?

## 7. Aliasing in discrete time

In discrete time, some frequencies are indistinguishable. For integer $t$,

$$
\cos(2\pi\omega t)=\cos(2\pi(1-\omega)t).
$$

Thus frequency $0.2$ and frequency $0.8$ produce the same sampled cosine values. This is called **aliasing**.

In [ ]:
t_alias = np.arange(30)
y_low = np.cos(2 * np.pi * 0.2 * t_alias)
y_high = np.cos(2 * np.pi * 0.8 * t_alias)

print("Maximum absolute difference:", np.max(np.abs(y_low - y_high)))

plt.figure()
plt.plot(t_alias, y_low, "o-", label="frequency 0.2")
plt.plot(t_alias, y_high, "x--", label="frequency 0.8")
plt.xlabel("time")
plt.ylabel("sampled value")
plt.title("Aliasing: two frequencies give the same sampled cosine")
plt.legend()
plt.show()

## 8. Spectral leakage

If a sinusoid completes an integer number of cycles in the observation window, its energy falls exactly on a Fourier frequency. If not, the energy spreads across nearby frequencies. This spreading is called **spectral leakage**.

We compare two signals:

- an on-grid sinusoid with frequency $12/240=0.05$;
- an off-grid sinusoid with frequency $12.5/240$, which is not exactly a Fourier frequency.

In [ ]:
n_leak = 240
t_leak = np.arange(n_leak)

x_on_grid = np.sin(2 * np.pi * (12 / n_leak) * t_leak)
x_off_grid = np.sin(2 * np.pi * (12.5 / n_leak) * t_leak)

f_on, p_on, _ = periodogram(x_on_grid)
f_off, p_off, _ = periodogram(x_off_grid)
pos_on = (f_on >= 0) & (f_on <= 0.5)
pos_off = (f_off >= 0) & (f_off <= 0.5)

plt.figure()
plt.plot(f_on[pos_on], p_on[pos_on], label="on-grid frequency")
plt.plot(f_off[pos_off], p_off[pos_off], label="off-grid frequency")
plt.xlabel("frequency")
plt.ylabel("periodogram")
plt.title("Spectral leakage")
plt.legend()
plt.show()

## 9. Tapering with a Hann window

A taper reduces sharp boundary discontinuities at the beginning and end of the observed window. Tapering can reduce leakage far away from the main frequency, but it also broadens the main peak.

The Hann taper is

$$
w_t = 0.5 - 0.5\cos\left(\frac{2\pi t}{n-1}\right).
$$

In [ ]:
hann = np.hanning(n_leak)
x_off_tapered = x_off_grid * hann

f_taper, p_taper, _ = periodogram(x_off_tapered)
pos_taper = (f_taper >= 0) & (f_taper <= 0.5)

plt.figure()
plt.plot(f_off[pos_off], p_off[pos_off] / np.max(p_off[pos_off]), label="off-grid, no taper")
plt.plot(f_taper[pos_taper], p_taper[pos_taper] / np.max(p_taper[pos_taper]), label="off-grid, Hann taper")
plt.xlabel("frequency")
plt.ylabel("normalized periodogram")
plt.title("Effect of tapering on spectral leakage")
plt.legend()
plt.show()

## 10. Smoothing the periodogram

The periodogram is useful for locating strong peaks, but it is noisy. A simple way to reduce variability is to average nearby frequencies.

This is similar in spirit to smoothing a noisy time series, but now the smoothing happens in the frequency domain.

In [ ]:
def moving_average(values, window):
    values = np.asarray(values, dtype=float)
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="same")

smoothed_power = moving_average(pos_power, window=7)

plt.figure()
plt.plot(pos_freqs, pos_power, label="raw periodogram", alpha=0.55)
plt.plot(pos_freqs, smoothed_power, label="smoothed periodogram", linewidth=2)
plt.xlabel("frequency")
plt.ylabel("power")
plt.title("Raw and smoothed periodogram")
plt.legend()
plt.show()

### Checkpoint 4

1. Does smoothing make the dominant frequency peaks easier to see?
2. What detail is lost when the periodogram is smoothed?
3. Why is the periodogram not usually treated as a perfectly smooth estimate of the population spectrum?

## 11. Empirical periodogram versus theoretical AR(1) spectrum

For a stationary AR(1) model

$$
X_t = \phi X_{t-1}+W_t,
$$

where $W_t$ is white noise with variance $\sigma^2$, the spectral shape is proportional to

$$
f(\omega)=\frac{\sigma^2}{1-2\phi\cos(2\pi\omega)+\phi^2}.
$$

If $\phi>0$, the series is smooth and low-frequency power is large. If $\phi<0$, the series alternates more rapidly and high-frequency power is larger.

In [ ]:
def simulate_ar1(phi, n=512, burnin=300, sigma=1.0, seed=123):
    local_rng = np.random.default_rng(seed)
    total = n + burnin
    w = local_rng.normal(0, sigma, size=total)
    x_full = np.zeros(total)
    for i in range(1, total):
        x_full[i] = phi * x_full[i - 1] + w[i]
    return x_full[burnin:]

def ar1_spectrum(freq, phi, sigma=1.0):
    return sigma ** 2 / (1 - 2 * phi * np.cos(2 * np.pi * freq) + phi ** 2)

x_ar_pos = simulate_ar1(phi=0.8, seed=10)
x_ar_neg = simulate_ar1(phi=-0.8, seed=11)

f_pos, p_pos, _ = periodogram(x_ar_pos)
f_neg, p_neg, _ = periodogram(x_ar_neg)
mask_pos = (f_pos >= 0) & (f_pos <= 0.5)
mask_neg = (f_neg >= 0) & (f_neg <= 0.5)

grid = np.linspace(0, 0.5, 400)
spec_pos = ar1_spectrum(grid, phi=0.8)
spec_neg = ar1_spectrum(grid, phi=-0.8)

# Normalize for visual comparison of shapes.
p_pos_norm = p_pos[mask_pos] / np.max(p_pos[mask_pos])
p_neg_norm = p_neg[mask_neg] / np.max(p_neg[mask_neg])
spec_pos_norm = spec_pos / np.max(spec_pos)
spec_neg_norm = spec_neg / np.max(spec_neg)

plt.figure()
plt.plot(f_pos[mask_pos], p_pos_norm, alpha=0.5, label="periodogram, phi = 0.8")
plt.plot(grid, spec_pos_norm, linewidth=2, label="theoretical shape, phi = 0.8")
plt.xlabel("frequency")
plt.ylabel("normalized power")
plt.title("AR(1) spectrum with positive autocorrelation")
plt.legend()
plt.show()

plt.figure()
plt.plot(f_neg[mask_neg], p_neg_norm, alpha=0.5, label="periodogram, phi = -0.8")
plt.plot(grid, spec_neg_norm, linewidth=2, label="theoretical shape, phi = -0.8")
plt.xlabel("frequency")
plt.ylabel("normalized power")
plt.title("AR(1) spectrum with negative autocorrelation")
plt.legend()
plt.show()

## 12. Practice: identify hidden periods automatically

The following helper function reports the largest periodogram peaks and translates them into estimated periods.

In [ ]:
def top_periodogram_peaks(values, num_peaks=6):
    freqs_local, power_local, _ = periodogram(values)
    mask = (freqs_local > 0) & (freqs_local <= 0.5)
    f = freqs_local[mask]
    p = power_local[mask]
    order = np.argsort(p)[-num_peaks:][::-1]
    result = pd.DataFrame({
        "frequency": f[order],
        "estimated_period": 1 / f[order],
        "periodogram": p[order]
    })
    return result

top_periodogram_peaks(x, num_peaks=8)

### Student task

Modify the synthetic time series below by changing the periods, amplitudes, and noise level. Then run the peak finder.

Try at least three cases:

1. Two clear periodic components with low noise.
2. Two periodic components with high noise.
3. Two periodic components whose frequencies are close together.

In [ ]:
# Student-editable experiment
n_exp = 300
t_exp = np.arange(n_exp)

period_a = 30
period_b = 10
amp_a = 1.5
amp_b = 0.7
noise_sd = 0.7

x_exp = (
    amp_a * np.sin(2 * np.pi * t_exp / period_a)
    + amp_b * np.cos(2 * np.pi * t_exp / period_b)
    + rng.normal(0, noise_sd, size=n_exp)
)

freq_exp, power_exp, _ = periodogram(x_exp)
mask_exp = (freq_exp >= 0) & (freq_exp <= 0.5)

plt.figure()
plt.plot(t_exp, x_exp)
plt.xlabel("time")
plt.ylabel("value")
plt.title("Student-editable synthetic series")
plt.show()

plt.figure()
plt.plot(freq_exp[mask_exp], power_exp[mask_exp])
plt.xlabel("frequency")
plt.ylabel("periodogram")
plt.title("Periodogram of student-editable series")
plt.show()

top_periodogram_peaks(x_exp, num_peaks=8)

## 13. AI-assisted study prompts

Use an AI assistant only after you have produced your own plots and written your own initial interpretation.

Suggested prompts:

1. **Concept check**  
   "I computed a periodogram for a time series. Explain how frequency and period are related, and why frequency 0 is special."

2. **Debugging prompt**  
   "Here is my code for a DFT and an FFT comparison. Check whether the normalization convention is consistent."

3. **Interpretation prompt**  
   "My periodogram has a large peak near frequency 0.083. What period does this correspond to, and how should I verify it in the time-domain plot?"

4. **Critical-thinking prompt**  
   "Explain why a periodogram can have many noisy local peaks even when the true spectrum is smooth."

Always ask the AI assistant to explain assumptions. Do not accept a spectral interpretation unless you can verify it using the time plot, frequency-to-period conversion, and model context.

## 14. Mini-project

Choose one of the following mini-projects.

### Option A: Hidden seasonality detector

Simulate a time series with three seasonal components and noise. Use the periodogram to recover the hidden periods. Explain which components are easy to detect and which are difficult.

### Option B: Leakage experiment

Compare periodograms for on-grid and off-grid sinusoidal frequencies. Show how the Hann taper changes the shape of the periodogram.

### Option C: AR spectral shapes

Simulate AR(1) processes for $\phi=-0.9,-0.5,0,0.5,0.9$. Compare their time-domain plots and periodograms. Explain how positive and negative autocorrelation appear in the frequency domain.

## Final checklist

Before leaving this lab, make sure you can do the following:

- [ ] Explain what a Fourier frequency is.
- [ ] Compute a DFT from the definition for a short sequence.
- [ ] Use `np.fft.fft` correctly with a normalization convention.
- [ ] Compute a periodogram and identify dominant frequencies.
- [ ] Convert a frequency into a period.
- [ ] Explain aliasing using sampled cosines.
- [ ] Explain spectral leakage and tapering.
- [ ] Compare empirical periodograms with theoretical AR(1) spectral shapes.